In [2]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

try:
    import seqeval
    import torchcrf
    import nervaluate
    import torch_geometric
    import spacy
except ImportError:
    print("Installing dependencies...")
    install("transformers>=4.35.0")
    install("datasets>=2.14.0")
    install("seqeval>=1.2.2")
    install("pytorch-crf>=0.7.2") 
    install("accelerate>=0.24.0")
    install("scikit-learn>=1.3.0")
    install("matplotlib>=3.7.0")
    install("pandas>=2.0.0")
    install("nervaluate")
    install("torch-geometric")
    install("spacy")
    
try:
    import spacy
    spacy.load("en_core_web_sm")
except Exception:
    print("Downloading spaCy en_core_web_sm model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("✅ All dependencies installed!")


✅ All dependencies installed!


In [3]:
import os
import torch

DATASET_TYPE = "ekstep"

USE_CONTEXT_WINDOW = True  
CONTEXT_WINDOW_SIZE = 3
USE_GAT = True             
USE_GATED_FUSION = False   

ENCODER_NAME = "evolawyer/inlegalbert-sc-ner-silver"
MAX_SEQ_LENGTH = 512          
LSTM_HIDDEN = 256
LSTM_LAYERS = 1
GAT_HIDDEN = 128
GAT_HEADS = 4
GAT_OUT = 512
DROPOUT = 0.3

ENCODER_LR = 2e-5
HEAD_LR = 1e-3
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EPOCHS = 5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
FREEZE_ENCODER_EPOCHS = 1
EARLY_STOPPING_PATIENCE = 2
SEED = 42
NUM_FOLDS = 1

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/SAIL_NER_v2_GAT_Only"
except Exception:
    if os.path.exists("/kaggle/working/"):
        SAVE_DIR = "/kaggle/working/SAIL_NER_v2_GAT_Only"
    else:
        SAVE_DIR = "./SAIL_NER_v2_GAT_Only"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Config loaded. Dataset: {DATASET_TYPE} | GAT: {USE_GAT} | Context window: {USE_CONTEXT_WINDOW}")
print(f"   Save directory: {SAVE_DIR}")
print(f"   GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


✅ Config loaded. Dataset: ekstep | GAT: True | Context window: True
   Save directory: ./SAIL_NER_v2_GAT_Only
   GPU available: True
   GPU: NVIDIA L4


In [4]:
if DATASET_TYPE == "il-tur":
    ENTITY_TYPES = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
else:
    ENTITY_TYPES = [
        "COURT", "PETITIONER", "RESPONDENT", "JUDGE", "LAWYER",
        "DATE", "ORG", "GPE", "STATUTE", "PROVISION",
        "PRECEDENT", "CASE_NUMBER", "WITNESS", "OTHER_PERSON"
    ]

LABEL_LIST = ["O"]
for ent in ENTITY_TYPES:
    LABEL_LIST.append(f"B-{ent}")
    LABEL_LIST.append(f"I-{ent}")

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

print(f"✅ Dataset Scheme: {DATASET_TYPE.upper()} | {NUM_LABELS} Tags")


✅ Dataset Scheme: EKSTEP | 29 Tags


In [5]:
import json
import zipfile
import requests
import re
import spacy
from spacy.tokens import Doc

DATA_DIR = os.path.join(SAVE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

if DATASET_TYPE == "ekstep":
    URLS = {
        "train": "https://storage.googleapis.com/indianlegalbert/OPEN_SOURCED_FILES/NER/NER_TRAIN.zip",
        "dev":   "https://storage.googleapis.com/indianlegalbert/OPEN_SOURCED_FILES/NER/NER_DEV.zip",
        "test":  "https://storage.googleapis.com/indianlegalbert/OPEN_SOURCED_FILES/NER/NER_TEST.zip",
    }
    def download_and_extract(url, name, data_dir):
        zip_path = os.path.join(data_dir, f"{name}.zip")
        extract_dir = os.path.join(data_dir, name)
        if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 0:
            return extract_dir
        print(f"   Downloading {name}...")
        r = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        os.remove(zip_path)
        return extract_dir

    for split, url in URLS.items():
        download_and_extract(url, split, DATA_DIR)
    print("✅ EkStep data loaded.")
else:
    LOCAL_IL_TUR_DIR = None
    candidates = ["./IL-TUR/dataset", "../IL-TUR/dataset", "./dataset", "/kaggle/input/il-tur/dataset"]
    for c in candidates:
        if os.path.exists(c):
            LOCAL_IL_TUR_DIR = c
            break
    if LOCAL_IL_TUR_DIR:
        print(f"✅ Found local IL-TUR dataset at: {LOCAL_IL_TUR_DIR}")
        DATA_DIR = LOCAL_IL_TUR_DIR
    else:
        print("⚠️ WARNING: Local IL-TUR dataset directory not found.")


✅ EkStep data loaded.


In [6]:
print("Loading spaCy for dependency parsing...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

def get_dependency_edges(words):
    """Generates syntactic dependency edges for a pre-tokenized list of words."""
    doc = Doc(nlp.vocab, words=words)
    doc = nlp(doc)
    edges = []
    for token in doc:
        if token.head.i != token.i:
            edges.append([token.i, token.head.i]) 
            edges.append([token.head.i, token.i]) 
    for i in range(len(words)):
        edges.append([i, i])
    return edges

def align_char_labels_to_words(text, spans):
    word_matches = list(re.finditer(r'\w+|[^\w\s]', text))
    words = [m.group(0) for m in word_matches]
    word_spans = [(m.start(), m.end()) for m in word_matches]
    labels = ['O'] * len(words)
    spans = sorted(spans, key=lambda x: x[0])
    for start, end, ent_type, ent_text in spans:
        if ent_type not in ENTITY_TYPES: continue
        first_word_idx = None
        for idx, (w_start, w_end) in enumerate(word_spans):
            if w_start >= start and w_end <= end:
                if first_word_idx is None:
                    first_word_idx = idx
                    labels[idx] = f"B-{ent_type}"
                else:
                    labels[idx] = f"I-{ent_type}"
    return words, labels

def split_document_into_sentences(words, labels):
    sentences = []
    curr_sent_words, curr_sent_labels = [], []
    abbreviations = {
        "mr", "mrs", "dr", "vs", "co", "ltd", "no", "nos", "art", "sec",
        "ors", "anr", "v", "p", "w", "crl", "app", "spl", "c", "o", "a",
        "i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x", "etc"
    }
    for idx, (word, label) in enumerate(zip(words, labels)):
        curr_sent_words.append(word)
        curr_sent_labels.append(label)
        if word in {".", "?", "!"}:
            is_abbr = (idx > 0 and words[idx - 1].lower() in abbreviations)
            is_followed_by_lower = (idx + 1 < len(words) and words[idx + 1][0].islower())
            if not is_abbr and not is_followed_by_lower:
                edges = get_dependency_edges(curr_sent_words) if USE_GAT else []
                sentences.append({
                    "words": curr_sent_words,
                    "labels": curr_sent_labels,
                    "edges": edges
                })
                curr_sent_words, curr_sent_labels = [], []
    if curr_sent_words:
        edges = get_dependency_edges(curr_sent_words) if USE_GAT else []
        sentences.append({
            "words": curr_sent_words,
            "labels": curr_sent_labels,
            "edges": edges
        })
    return sentences

def parse_spacy_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    documents = []
    if isinstance(data, dict): data = [data]
    for doc in data:
        if "annotations" in doc and "data" in doc and "text" in doc["data"]:
            text = doc["data"]["text"]
            spans = []
            for annot in doc.get("annotations", []):
                for res in annot.get("result", []):
                    if res.get("type") == "labels":
                        val = res.get("value", {})
                        if "start" in val and "end" in val and "labels" in val:
                            spans.append((val["start"], val["end"], val["labels"][0], val.get("text","")))
            words, labels = align_char_labels_to_words(text, spans)
            sentences = split_document_into_sentences(words, labels)
            documents.append(sentences)
        else:
            doc_sentences = []
            paragraphs = doc.get("paragraphs", doc.get("annotations", []))
            if not paragraphs and "sentences" in doc: paragraphs = [doc]
            for para in paragraphs:
                sentences = para.get("sentences", [])
                if not sentences:
                    tokens = para.get("tokens", [])
                    if tokens: sentences = [{"tokens": tokens}]
                for sent in sentences:
                    tokens = sent.get("tokens", [])
                    sent_data = []
                    for tok in tokens:
                        text = tok.get("orth", tok.get("text", ""))
                        ner_tag = tok.get("ner", tok.get("entity", "O"))
                        if ner_tag in ("", "-", "O", "0"): ner_tag = "O"
                        if ner_tag.startswith("U-"): ner_tag = "B-" + ner_tag[2:]
                        elif ner_tag.startswith("L-"): ner_tag = "I-" + ner_tag[2:]
                        if ner_tag != "O" and ner_tag not in LABEL2ID: ner_tag = "O"
                        if text.strip(): sent_data.append((text, ner_tag))
                    if sent_data: doc_sentences.append(sent_data)
            if doc_sentences:
                adapted_sentences = []
                for sent_data in doc_sentences:
                    words = [w for w, l in sent_data]
                    labels = [l for w, l in sent_data]
                    edges = get_dependency_edges(words) if USE_GAT else []
                    adapted_sentences.append({"words": words, "labels": labels, "edges": edges})
                documents.append(adapted_sentences)
    return documents

def load_ekstep_split(split_name):
    cache_path = os.path.join(DATA_DIR, f"{split_name}_cache.pt")
    if os.path.exists(cache_path):
        print(f"Loading cached {split_name} data from {cache_path}")
        return torch.load(cache_path)
        
    print(f"Parsing {split_name} (this may take a few minutes for spaCy parsing)...")
    split_dir = os.path.join(DATA_DIR, split_name)
    json_files = sorted([os.path.join(r, f) for r, d, files in os.walk(split_dir) for f in files if f.endswith(".json")])
    all_documents = []
    for jf in json_files:
        all_documents.extend(parse_spacy_json(jf))
        
    torch.save(all_documents, cache_path)
    return all_documents

def load_il_tur_file(filepath):
    cache_path = filepath + "_cache.pt"
    if os.path.exists(cache_path):
        print(f"Loading cached IL-TUR data from {cache_path}")
        return torch.load(cache_path)
        
    print(f"Parsing {filepath} (this may take a few minutes for spaCy parsing)...")
    with open(filepath, 'r', encoding='utf-8') as f: data = json.load(f)
    documents = []
    for doc in data['data']:
        words, labels = align_char_labels_to_words(doc['text'], doc['labels'])
        sentences = split_document_into_sentences(words, labels)
        documents.append(sentences)
        
    torch.save(documents, cache_path)
    return documents

def get_il_tur_splits(fold_idx, dataset_dir):
    files = ["1.json", "2.json", "3.json"]
    test_file = files[fold_idx]
    train_files = [f for f in files if f != test_file]
    train_docs = []
    for tf in train_files: train_docs.extend(load_il_tur_file(os.path.join(dataset_dir, tf)))
    test_docs = load_il_tur_file(os.path.join(dataset_dir, test_file))
    
    import random
    random.seed(42)
    random.shuffle(train_docs)
    dev_size = int(0.2 * len(train_docs))
    dev_docs = train_docs[-dev_size:]
    actual_train_docs = train_docs[:-dev_size]
    return actual_train_docs, dev_docs, test_docs


Loading spaCy for dependency parsing...


In [7]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

class LegalNERDataset(Dataset):
    def __init__(self, documents, use_ctx=True):
        self.examples = []
        for doc in documents:
            for i in range(len(doc)):
                target_sent = doc[i]
                if use_ctx:
                    prev_sent = doc[i-1] if i > 0 else {"words": []}
                    next_sent = doc[i+1] if i < len(doc)-1 else {"words": []}
                else:
                    prev_sent = {"words": []}
                    next_sent = {"words": []}
                    
                self.examples.append({
                    "prev_words": prev_sent["words"],
                    "target_words": target_sent["words"],
                    "target_labels": target_sent["labels"],
                    "target_edges": target_sent["edges"],
                    "next_words": next_sent["words"],
                })
        print(f"  Created {len(self.examples)} examples")

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

def get_collate_fn(tokenizer, label2id, max_length=MAX_SEQ_LENGTH):
    def collate(batch):
        processed_examples = []
        max_target_words = 0
        
        for ex in batch:
            prev_words = ex["prev_words"]
            target_words = ex["target_words"]
            target_labels = ex["target_labels"]
            next_words = ex["next_words"]
            
            target_subwords = []
            for w in target_words: target_subwords.extend(tokenizer.tokenize(w))
                
            budget = max(0, max_length - len(target_subwords) - 2)
            half_budget = budget // 2
            
            prev_subwords_count = 0
            truncated_prev_words = []
            for w in reversed(prev_words):
                sub_len = len(tokenizer.tokenize(w))
                if prev_subwords_count + sub_len <= half_budget:
                    truncated_prev_words.insert(0, w)
                    prev_subwords_count += sub_len
                else: break
                    
            next_subwords_count = 0
            truncated_next_words = []
            rem_budget = budget - prev_subwords_count
            for w in next_words:
                sub_len = len(tokenizer.tokenize(w))
                if next_subwords_count + sub_len <= rem_budget:
                    truncated_next_words.append(w)
                    next_subwords_count += sub_len
                else: break
                    
            input_words = truncated_prev_words + target_words + truncated_next_words
            actual_prev_len = len(truncated_prev_words)
            
            encoding = tokenizer(
                input_words, is_split_into_words=True, max_length=max_length,
                padding="max_length", truncation=True, return_tensors="pt"
            )
            
            word_ids = encoding.word_ids(batch_index=0)
            target_word_indices = []
            target_word_labels = []
            
            prev_word_id = None
            for idx, word_id in enumerate(word_ids):
                if word_id is None: continue
                if word_id != prev_word_id:
                    if actual_prev_len <= word_id < actual_prev_len + len(target_words):
                        target_word_indices.append(idx)
                        target_idx = word_id - actual_prev_len
                        label_str = target_labels[target_idx]
                        target_word_labels.append(label2id.get(label_str, label2id["O"]))
                prev_word_id = word_id
                
            if not target_word_indices:
                target_word_indices = [0]
                target_word_labels = [label2id["O"]]
                
            max_target_words = max(max_target_words, len(target_word_indices))
            
            processed_examples.append({
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "target_word_indices": target_word_indices,
                "labels": target_word_labels,
                "target_edges": ex["target_edges"]
            })
            
        padded_input_ids, padded_attention_mask = [], []
        padded_word_indices, padded_labels, padded_word_mask = [], [], []
        batch_edge_indices = []
        
        for i, ex in enumerate(processed_examples):
            padded_input_ids.append(ex["input_ids"])
            padded_attention_mask.append(ex["attention_mask"])
            
            indices = ex["target_word_indices"]
            labels = ex["labels"]
            pad_len = max_target_words - len(indices)
            
            padded_word_indices.append(torch.tensor(indices + [0] * pad_len, dtype=torch.long))
            padded_labels.append(torch.tensor(labels + [-100] * pad_len, dtype=torch.long))
            padded_word_mask.append(torch.tensor([True] * len(indices) + [False] * pad_len, dtype=torch.bool))
            
            if USE_GAT and ex["target_edges"]:
                valid_edges = []
                for u, v in ex["target_edges"]:
                    if u < len(indices) and v < len(indices):
                        valid_edges.append([u, v])
                if valid_edges:
                    edges = torch.tensor(valid_edges, dtype=torch.long)
                    offset = i * max_target_words
                    batch_edge_indices.append(edges + offset)
                
        if USE_GAT and batch_edge_indices:
            padded_edge_index = torch.cat(batch_edge_indices, dim=0).t().contiguous()
        else:
            padded_edge_index = torch.empty((2, 0), dtype=torch.long)
            
        return {
            "input_ids": torch.stack(padded_input_ids),
            "attention_mask": torch.stack(padded_attention_mask),
            "target_word_indices": torch.stack(padded_word_indices),
            "target_word_mask": torch.stack(padded_word_mask),
            "labels": torch.stack(padded_labels),
            "edge_index": padded_edge_index
        }
    return collate


In [8]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torchcrf import CRF

try:
    from torch_geometric.nn import GATConv
except ImportError:
    if USE_GAT:
        raise RuntimeError("torch_geometric is required when USE_GAT=True. Install it first.")
    GATConv = None

class SyntacticGAT(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=GAT_HIDDEN, out_dim=GAT_OUT, heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.gat2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        if edge_index.numel() == 0:
            return torch.zeros((x.size(0), self.gat2.out_channels), device=x.device)
        x = F.elu(self.gat1(x, edge_index))
        x = self.dropout(x)
        x = self.gat2(x, edge_index)
        return x

class GatedFusion(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.gate = nn.Linear(dim * 2, dim)

    def forward(self, h_seq, h_syn):
        g = torch.sigmoid(self.gate(torch.cat([h_seq, h_syn], dim=-1)))
        return g * h_seq + (1 - g) * h_syn

class SAILNER_v2(nn.Module):
    def __init__(self, encoder_name=ENCODER_NAME, num_labels=NUM_LABELS):
        super().__init__()
        self.use_gat = USE_GAT
        self.use_gated_fusion = USE_GATED_FUSION
        
        self.encoder = AutoModel.from_pretrained(encoder_name)
        enc_dim = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(DROPOUT)
        
        self.gat = SyntacticGAT(in_dim=enc_dim, out_dim=GAT_OUT)
        classifier_in_dim = GAT_OUT
            
        self.classifier = nn.Linear(classifier_in_dim, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, target_word_indices, target_word_mask, edge_index=None, labels=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        
        expanded_indices = target_word_indices.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        word_reprs = torch.gather(hidden, dim=1, index=expanded_indices)
        word_reprs = word_reprs * target_word_mask.unsqueeze(-1)
        word_reprs = self.dropout(word_reprs)
        
        b, seq_len, h = word_reprs.size()
        flat_word_reprs = word_reprs.view(b * seq_len, h)
        gat_out_flat = self.gat(flat_word_reprs, edge_index)
        gat_out = gat_out_flat.view(b, seq_len, -1)
        gat_out = gat_out * target_word_mask.unsqueeze(-1)
        fused_out = gat_out
        emissions = self.classifier(fused_out)

        if labels is not None:
            crf_labels = labels.clone()
            crf_labels[labels == -100] = 0
            loss = -self.crf(emissions, crf_labels, mask=target_word_mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=target_word_mask)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [9]:
import time
import numpy as np
import random
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(SEED)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            t_idx = batch["target_word_indices"].to(device)
            t_mask = batch["target_word_mask"].to(device)
            edges = batch["edge_index"].to(device)
            labs = batch["labels"]
            
            preds = model(ids, mask, t_idx, t_mask, edges)
            for i in range(len(preds)):
                pred_ids = preds[i]
                label_ids = [l for l in labs[i].tolist() if l != -100]
                pt = [ID2LABEL.get(p, "O") for p in pred_ids]
                tt = [ID2LABEL.get(t, "O") for t in label_ids]
                if tt:
                    all_preds.append(pt)
                    all_labels.append(tt)
    try:
        macro = seqeval_f1(all_labels, all_preds, mode="strict", scheme=IOB2, average="macro")
        micro = seqeval_f1(all_labels, all_preds, mode="strict", scheme=IOB2, average="micro")
        report = classification_report(all_labels, all_preds, mode="strict", scheme=IOB2)
    except:
        macro = seqeval_f1(all_labels, all_preds, average="macro")
        micro = seqeval_f1(all_labels, all_preds, average="micro")
        report = classification_report(all_labels, all_preds)
    return macro, micro, report, all_preds, all_labels


In [10]:
def freeze_encoder(m):
    for p in m.encoder.parameters(): p.requires_grad = False
    print("  Encoder FROZEN")
def unfreeze_encoder(m):
    for p in m.encoder.parameters(): p.requires_grad = True
    print("  Encoder UNFROZEN")

def train_fold(fold_idx, train_docs, dev_docs, test_docs):
    print(f"\n==================== TRAINING FOLD {fold_idx} ====================")
    train_dataset = LegalNERDataset(train_docs, use_ctx=USE_CONTEXT_WINDOW)
    dev_dataset   = LegalNERDataset(dev_docs,   use_ctx=USE_CONTEXT_WINDOW)
    
    collate_fn = get_collate_fn(tokenizer, LABEL2ID, MAX_SEQ_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, pin_memory=True)
    dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn, pin_memory=True)
    
    model = SAILNER_v2(num_labels=NUM_LABELS).to(device)
    
    head_params = list(model.classifier.parameters()) + list(model.crf.parameters())
    if USE_GAT:
        head_params.extend(model.gat.parameters())
        if USE_GATED_FUSION:
            head_params.extend(model.fusion.parameters())
            
    optimizer = AdamW([
        {"params": [p for _, p in model.encoder.named_parameters() if p.requires_grad], "lr": ENCODER_LR, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
    ])
    
    total_steps = (len(train_loader) // GRADIENT_ACCUMULATION) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    
    best_f1, patience_counter = 0.0, 0
    history = {"train_loss": [], "dev_macro_f1": [], "dev_micro_f1": []}
    
    for epoch in range(EPOCHS):
        t0 = time.time()
        if epoch < FREEZE_ENCODER_EPOCHS: freeze_encoder(model)
        elif epoch == FREEZE_ENCODER_EPOCHS: unfreeze_encoder(model)
            
        model.train()
        total_loss, n_batch = 0.0, 0
        optimizer.zero_grad()
        
        for step, batch in enumerate(train_loader):
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            t_idx = batch["target_word_indices"].to(device)
            t_mask = batch["target_word_mask"].to(device)
            edges = batch["edge_index"].to(device)
            labs = batch["labels"].to(device)
            
            loss = model(ids, mask, t_idx, t_mask, edges, labs) / GRADIENT_ACCUMULATION
            loss.backward()
            total_loss += loss.item() * GRADIENT_ACCUMULATION
            n_batch += 1
            
            if (step + 1) % GRADIENT_ACCUMULATION == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                
            if (step + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/n_batch:.4f}")
                
        if len(train_loader) % GRADIENT_ACCUMULATION != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        avg_loss = total_loss / n_batch
        macro_f1, micro_f1, _, _, _ = evaluate(model, dev_loader, device)
        history["train_loss"].append(avg_loss)
        history["dev_macro_f1"].append(macro_f1)
        history["dev_micro_f1"].append(micro_f1)
        
        print(f"  Epoch {epoch+1} done in {time.time()-t0:.1f}s | Loss: {avg_loss:.4f} | Dev Macro-F1: {macro_f1:.4f}")
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            patience_counter = 0
            sp = os.path.join(SAVE_DIR, f"sail_ner_v2_fold_{fold_idx}.pt")
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "best_f1": best_f1}, sp)
            print(f"  💾 Saved best model checkpoint to {sp}")
        else:
            patience_counter += 1
            print(f"  ⏳ Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print("  ⛔ Early stopping triggered.")
                break
    return history

all_histories = []
if DATASET_TYPE == "il-tur":
    for fold in range(NUM_FOLDS):
        train_d, dev_d, test_d = get_il_tur_splits(fold, DATA_DIR)
        all_histories.append(train_fold(fold, train_d, dev_d, test_d))
else:
    print("Loading EkStep Dataset splits...")
    train_docs = load_ekstep_split("train")
    dev_docs   = load_ekstep_split("dev")
    test_docs  = load_ekstep_split("test")
    all_histories.append(train_fold(0, train_docs, dev_docs, test_docs))


Loading EkStep Dataset splits...
Loading cached train data from ./SAIL_NER_v2_GAT_Only/data/train_cache.pt


Loading cached dev data from ./SAIL_NER_v2_GAT_Only/data/dev_cache.pt
Loading cached test data from ./SAIL_NER_v2_GAT_Only/data/test_cache.pt

==================== TRAINING FOLD 0 ====================
  Created 55321 examples
  Created 11470 examples


Some weights of BertModel were not initialized from the model checkpoint at evolawyer/inlegalbert-sc-ner-silver and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Encoder FROZEN
  Epoch 1/5 | Step 50/6916 | Loss: 37.3609
  Epoch 1/5 | Step 100/6916 | Loss: 31.3858
  Epoch 1/5 | Step 150/6916 | Loss: 24.7264
  Epoch 1/5 | Step 200/6916 | Loss: 21.3236
  Epoch 1/5 | Step 250/6916 | Loss: 18.6767
  Epoch 1/5 | Step 300/6916 | Loss: 16.9956
  Epoch 1/5 | Step 350/6916 | Loss: 15.7986
  Epoch 1/5 | Step 400/6916 | Loss: 14.7765
  Epoch 1/5 | Step 450/6916 | Loss: 13.8246
  Epoch 1/5 | Step 500/6916 | Loss: 12.9645
  Epoch 1/5 | Step 550/6916 | Loss: 12.2362
  Epoch 1/5 | Step 600/6916 | Loss: 11.6647
  Epoch 1/5 | Step 650/6916 | Loss: 11.1612
  Epoch 1/5 | Step 700/6916 | Loss: 10.7003
  Epoch 1/5 | Step 750/6916 | Loss: 10.4226
  Epoch 1/5 | Step 800/6916 | Loss: 10.1668
  Epoch 1/5 | Step 850/6916 | Loss: 9.8885
  Epoch 1/5 | Step 900/6916 | Loss: 9.5939
  Epoch 1/5 | Step 950/6916 | Loss: 9.3348
  Epoch 1/5 | Step 1000/6916 | Loss: 9.1224
  Epoch 1/5 | Step 1050/6916 | Loss: 8.9389
  Epoch 1/5 | Step 1100/6916 | Loss: 8.7936
  Epoch 1/5 | Step 

In [10]:
from nervaluate import Evaluator

def test_evaluation():
    fold_macro_f1s = []
    
    if DATASET_TYPE == "il-tur":
        for fold in range(NUM_FOLDS):
            print(f"\n--- EVALUATING FOLD {fold} ---")
            _, _, test_docs = get_il_tur_splits(fold, DATA_DIR)
            test_dataset = LegalNERDataset(test_docs, use_ctx=USE_CONTEXT_WINDOW)
            collate_fn = get_collate_fn(tokenizer, LABEL2ID, MAX_SEQ_LENGTH)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn, pin_memory=True)
            
            model = SAILNER_v2(num_labels=NUM_LABELS).to(device)
            ckpt_path = os.path.join(SAVE_DIR, f"sail_ner_v2_fold_{fold}.pt")
            if os.path.exists(ckpt_path):
                ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
                model.load_state_dict(ckpt["model_state_dict"])
                print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
                
            macro, micro, report, all_preds, all_labels = evaluate(model, test_loader, device)
            print(report)
            print(f"Fold {fold} Test Macro-F1: {macro:.4f}")
            fold_macro_f1s.append(macro)
            
            with open(os.path.join(SAVE_DIR, f"results_fold_{fold}_v2.txt"), "w") as f: f.write(report)
            evaluator = Evaluator(all_labels, all_preds, tags=ENTITY_TYPES, loader='list')
            results, results_per_tag, *_ = evaluator.evaluate()
            with open(os.path.join(SAVE_DIR, f"nervaluate_fold_{fold}_v2.json"), "w") as fj:
                json.dump({"overall": results, "label_wise": results_per_tag}, fj, indent=4)
                
        if len(fold_macro_f1s) > 1:
            print(f"\n============================================================")
            print(f"  CROSS VALIDATION MEAN MACRO-F1: {np.mean(fold_macro_f1s):.4f} (std={np.std(fold_macro_f1s):.4f})")
            print(f"============================================================")
    else:
        print("\n--- EVALUATING EKSTEP TEST SET ---")
        test_docs = load_ekstep_split("test")
        test_dataset = LegalNERDataset(test_docs, use_ctx=USE_CONTEXT_WINDOW)
        collate_fn = get_collate_fn(tokenizer, LABEL2ID, MAX_SEQ_LENGTH)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn, pin_memory=True)
        
        model = SAILNER_v2(num_labels=NUM_LABELS).to(device)
        ckpt_path = os.path.join(SAVE_DIR, "sail_ner_v2_fold_0.pt")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"])
            print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
            
        macro, micro, report, all_preds, all_labels = evaluate(model, test_loader, device)
        print(report)
        print(f"EkStep Test Macro-F1: {macro:.4f}")
        
        with open(os.path.join(SAVE_DIR, "results_ekstep_v2.txt"), "w") as f: f.write(report)
        evaluator = Evaluator(all_labels, all_preds, tags=ENTITY_TYPES, loader='list')
        results, results_per_tag, *_ = evaluator.evaluate()
        with open(os.path.join(SAVE_DIR, "nervaluate_ekstep_v2.json"), "w") as fj:
            json.dump({"overall": results, "label_wise": results_per_tag}, fj, indent=4)

test_evaluation()



--- EVALUATING EKSTEP TEST SET ---
Loading cached test data from ./SAIL_NER_v2_GAT_Only/data/test_cache.pt


  Created 23782 examples


Some weights of BertModel were not initialized from the model checkpoint at evolawyer/inlegalbert-sc-ner-silver and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded checkpoint from epoch 5
              precision    recall  f1-score   support

 CASE_NUMBER       0.73      0.72      0.73       788
       COURT       0.92      0.91      0.92      1510
        DATE       0.86      0.89      0.88      1330
         GPE       0.74      0.76      0.75       900
       JUDGE       0.91      0.97      0.94       760
      LAWYER       0.96      0.94      0.95      2386
         ORG       0.77      0.75      0.76      1073
OTHER_PERSON       0.85      0.84      0.84      1381
  PETITIONER       0.78      0.76      0.77      1073
   PRECEDENT       0.74      0.77      0.75       823
   PROVISION       0.92      0.93      0.92      1472
  RESPONDENT       0.72      0.74      0.73      1427
     STATUTE       0.90      0.92      0.91      1183
     WITNESS       0.89      0.92      0.90       471

   micro avg       0.85      0.85      0.85     16577
   macro avg       0.84      0.84      0.84     16577
weighted avg       0.85      0.85      0.85     1

In [12]:
def predict_entities(text, model, tokenizer, device):
    model.eval()
    words = text.split()
    
    encoding = tokenizer(
        words, is_split_into_words=True, max_length=MAX_SEQ_LENGTH,
        padding="max_length", truncation=True, return_tensors="pt"
    )
    
    ids = encoding["input_ids"].to(device)
    mask = encoding["attention_mask"].to(device)
    word_ids = encoding.word_ids(batch_index=0)
    
    target_word_indices = []
    prev_word_id = None
    for idx, word_id in enumerate(word_ids):
        if word_id is None: continue
        if word_id != prev_word_id:
            target_word_indices.append(idx)
        prev_word_id = word_id
        
    target_word_mask = [True] * len(target_word_indices)
    t_target_word_indices = torch.tensor([target_word_indices], dtype=torch.long).to(device)
    t_target_word_mask = torch.tensor([target_word_mask], dtype=torch.bool).to(device)
    
    edges = get_dependency_edges(words) if USE_GAT else []
    t_edges = torch.tensor(edges, dtype=torch.long).t().contiguous().to(device) if edges else torch.empty((2,0), dtype=torch.long).to(device)
    
    with torch.no_grad():
        preds = model(ids, mask, t_target_word_indices, t_target_word_mask, t_edges)
        
    pred_ids = preds[0]
    results = []
    for w, p_id in zip(words, pred_ids):
        results.append((w, ID2LABEL.get(p_id, "O")))
    return results

model = SAILNER_v2(num_labels=NUM_LABELS).to(device)
ckpt_path = os.path.join(SAVE_DIR, "sail_ner_v2_fold_0.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    
demos = [
    "The petitioner Rajesh Kumar filed a writ petition under Article 226 of the Constitution in the High Court of Delhi",
    "In the case of State of Maharashtra vs Mohd Yakub 1980 AIR 1111 the Supreme Court held that Section 302 of IPC applies",
    "Justice D.Y. Chandrachud delivered the judgment on 15th March 2023 in New Delhi",
]
print("SAIL-NER v2 Inference Demo\n" + "=" * 60)
for text in demos:
    print(f"\nInput: {text}")
    for w, l in predict_entities(text, model, tokenizer, device):
        if l != "O": print(f"  {l:20s} -> {w}")
print("\nDone!")


Some weights of BertModel were not initialized from the model checkpoint at evolawyer/inlegalbert-sc-ner-silver and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SAIL-NER v2 Inference Demo

Input: The petitioner Rajesh Kumar filed a writ petition under Article 226 of the Constitution in the High Court of Delhi
  B-PETITIONER         -> Rajesh
  I-PETITIONER         -> Kumar
  B-PROVISION          -> Article
  I-PROVISION          -> 226
  B-STATUTE            -> Constitution
  B-COURT              -> High
  I-COURT              -> Court
  I-COURT              -> of
  I-COURT              -> Delhi

Input: In the case of State of Maharashtra vs Mohd Yakub 1980 AIR 1111 the Supreme Court held that Section 302 of IPC applies
  B-PRECEDENT          -> State
  I-PRECEDENT          -> of
  I-PRECEDENT          -> Maharashtra
  I-PRECEDENT          -> vs
  I-PRECEDENT          -> Mohd
  I-PRECEDENT          -> Yakub
  I-PRECEDENT          -> 1980
  I-PRECEDENT          -> AIR
  I-PRECEDENT          -> 1111
  B-COURT              -> Supreme
  I-COURT              -> Court
  B-PROVISION          -> Section
  I-PROVISION          -> 302
  B-STATUTE       